In [4]:
!pip install matplotlib -q

In [1]:
import ray
from ray import serve
from ray.llm._internal.serve.deployments.routers.prefix_tree import PrefixTree
from ray.serve.handle import DeploymentHandle, DeploymentResponse
import time
import matplotlib.pyplot as plt
import numpy as np
import asyncio

TypeError: 'module' object is not callable

In [6]:

tree_deployment = PrefixTree.bind()
tree_handle: DeploymentHandle = serve.run(tree_deployment, route_prefix="/hello")
hello_response = await tree_handle.hello.remote()
print(hello_response)

AttributeError: type object 'PrefixTree' has no attribute 'bind'

In [ ]:

def benchmark_deployment_calls(handle=None, num_calls=0):
    start_time = time.time()
    for _ in range(num_calls):
        handle.hello.remote()
    end_time = time.time()
    return end_time - start_time
def benchmark_deployment_overhead(handle=None, min_calls=0, max_calls=100000, step_size=25000):
    # Generate call counts from min to max with the given step size
    call_counts = [0] if min_calls == 0 else []
    call_counts.extend(list(range(min_calls, max_calls + 1, step_size)))
    times = []

    # Run the benchmark for each number of calls
    for num_calls in call_counts:
        if num_calls == 0:
            times.append(0)  # No calls takes no time
            continue
        
        print(f"Benchmarking {num_calls} calls...")
        elapsed_time = benchmark_deployment_calls(handle, num_calls)
        times.append(elapsed_time)
        print(f"Time for {num_calls} calls: {elapsed_time:.2f} seconds")

    # Create the plot
    plt.figure(figsize=(10, 6))
    plt.plot(call_counts, times, marker='o', linestyle='-', linewidth=2)
    plt.title('Deployment Call Overhead')
    plt.xlabel('Number of Calls')
    plt.ylabel('Time (seconds)')
    plt.grid(True)

    # Add a best fit line
    if len(times) > 1:
        z = np.polyfit(call_counts, times, 1)
        p = np.poly1d(z)
        plt.plot(call_counts, p(call_counts), "r--", alpha=0.7, 
                 label=f"Trend: {z[0]*1000:.3f} ms per call")
        plt.legend()

    plt.tight_layout()
    plt.show()

    # Calculate and display the average time per call
    if len(times) > 1 and call_counts[-1] > 0:
        avg_time_per_call = times[-1] / call_counts[-1]
        print(f"Average time per call: {avg_time_per_call*1000:.3f} milliseconds")
    
    return call_counts, times
import os
import sys
from contextlib import redirect_stdout

with open(os.devnull, 'w') as devnull:
    with redirect_stdout(devnull):
        benchmark_deployment_overhead(tree_handle, min_calls=0, max_calls=1000, step_size=10)